##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


#### **Import Required Libraries:**
Start by importing all required libraries.

In [29]:
import os
import cv2
import math
import pafy
import random
import numpy as np
import datetime as dt
import tensorflow as tf
from moviepy.editor import *
from collections import deque
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split

from tensorflow.keras.layers import *
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import plot_model

**Set Numpy, Python & Tensorflow seeds to get consistent results.**

In [30]:
seed_constant = 23
np.random.seed(seed_constant)
random.seed(seed_constant)
tf.random.set_seed(seed_constant)

## **Step 1: Download the Dataset**

Let’s start by downloading the dataset. 

The Dataset we are using is the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php).




In [31]:
import os

dataset_path = "action_youtube_naudio"
print(os.listdir(dataset_path))

['volleyball_spiking', 'golf_swing', 'swing', 'tennis_swing', 'soccer_juggling', 'horse_riding', 'walking', 'basketball', 'trampoline_jumping', 'biking', 'readme.txt', 'diving']


## **Step 3: Read & Preprocess the Dataset**

Since we're going to use a classification architecture to train on a video classification dataset, we're going to need to preprocess the dataset first.

Now with the constants, 
- **`image_height`** and **`image_weight`**: This is the size we will resize all frames of the video to, we're doing this to avoid unneccsary computation.

- **`max_images_per_class`**: Maximum number of training images allowed for each class.

- **`dataset_directory`**: The path of the directory containing the extracted dataset. 

- **`classes_list`**: These are the list of classes we're going to be training on, we're traninng on following 4 classes, you can feel free to change it. 
  - *tai chi*
  - *Swinging*
  - *Horse Racing*
  - *Walking with a Dog*

**Note:** The `image_height`, `image_weight` and `max_images_per_class` constants may be increased for better results, but be warned this will become computationally expensive.

In [32]:
image_height, image_width = 64, 64
max_images_per_class = 1000

dataset_directory = "action_youtube_naudio"
classes_list = ['walking', 'basketball', 'trampoline_jumping']

model_output_size = len(classes_list)

### **Extract, Resize & Normalize Frames**


Now we'll create a function that will extract frames from each video while performing other preprocessing operation like resizing and normalizing images. 

This method takes a video file path as input. It then reads the video file frame by frame, resizes each frame, normalizes the resized frame, appends the normalized frame into a list and then finally returns that list.

In [33]:
def frames_extraction(video_path):
    # Empty List declared to store video frames
    frames_list = []
    
    # Reading the Video File Using the VideoCapture
    video_reader = cv2.VideoCapture(video_path)

    # Iterating through Video Frames
    while True:

        # Reading a frame from the video file 
        success, frame = video_reader.read() 

        # If Video frame was not successfully read then break the loop
        if not success:
            break

        # Resize the Frame to fixed Dimensions
        resized_frame = cv2.resize(frame, (image_height, image_width))
        
        # Normalize the resized frame by dividing it with 255 so that each pixel value then lies between 0 and 1
        normalized_frame = resized_frame / 255
        
        # Appending the normalized frame into the frames list
        frames_list.append(normalized_frame)
    
    # Closing the VideoCapture object and releasing all resources. 
    video_reader.release()

    # returning the frames list 
    return frames_list

### **Dataset Creation**
Now we'll create another function called  **`create_dataset()`**,  this function uses the **`frame_extraction()`** funciton above and creates our final preprocessed dataset. 

**Here's how this function works:**

1.   Iterate through all the classes mentioned in the `classes_list`
2.   Now for each class iterate through all the video files present in it. 
3.   Call the **frame_extraction** method on each video file.
4.   Add the returned frames to a list called `temp_features`
5.   After all videos of a class are processed, randomly select video frames (equal to **max_images_per_class**) and add them to the list called `features`.
6.   Add labels of the selected videos to the labels list.
7.   After all videos of all classes are processed then return the features and labels as numpy arrays.


So when you call this function, it returns **2** lists:
- a list of feature vectors 
- a list of it's associated labels.


In [34]:
import glob

def create_dataset():
    features = []
    labels = []

    for class_index, class_name in enumerate(classes_list):
        print(f"Extracting Data of Class: {class_name}")

        class_path = os.path.join(dataset_directory, class_name)

        video_paths = glob.glob(os.path.join(class_path, "**", "*.avi"), recursive=True)

        temp_features = []

        for video_file_path in video_paths:
            frames = frames_extraction(video_file_path)
            temp_features.extend(frames)

        print("Total frames:", len(temp_features))

        sample_size = min(max_images_per_class, len(temp_features))

        features.extend(random.sample(temp_features, sample_size))
        labels.extend([class_index] * sample_size)

    features = np.asarray(features)
    labels = np.array(labels)

    return features, labels

Calling the **create_dataset** method which returns features and labels.

In [35]:
features, labels = create_dataset()
print(features.shape)
print(labels.shape)

Extracting Data of Class: walking
Total frames: 23496
Extracting Data of Class: basketball
Total frames: 15092
Extracting Data of Class: trampoline_jumping
Total frames: 21578
(3000, 64, 64, 3)
(3000,)


Now we will convert class labels to one hot encoded vectors.

In [36]:
# Using Keras's to_categorical method to convert labels into one-hot-encoded vectors
one_hot_encoded_labels = to_categorical(labels)

## **Step 4: Split the Data into Train and Test Set**
Now we have 2 numpy arrays, one containing all images, the second one contains all class labels in one hot encoded format. Let’s split our data to create a training and a testing set. It’s important that you shuffle your data before the split which we have already done.


In [37]:
features_train, features_test, labels_train, labels_test = train_test_split(features, one_hot_encoded_labels, test_size = 0.2, shuffle = True, random_state = seed_constant)

## **Step 5: Construct the Model**
Now it’s time to create our CNN model, for this post, we're c reating a simple CNN Classification model with two CNN layers.

In [38]:
# Let's create a function that will construct our model
def create_model():

    # We will use a Sequential model for model construction
    model = Sequential()

    # Defining The Model Architecture
    model.add(Conv2D(filters = 64, kernel_size = (3, 3), activation = 'relu', input_shape = (image_height, image_width, 3)))
    model.add(Conv2D(filters = 64, kernel_size = (3, 3), activation = 'relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size = (2, 2)))
    model.add(GlobalAveragePooling2D())
    model.add(Dense(256, activation = 'relu'))
    model.add(BatchNormalization())
    model.add(Dense(model_output_size, activation = 'softmax'))

    # Printing the models summary
    model.summary()

    return model


# Calling the create_model method
model = create_model()

print("Model Created Successfully!")

  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 62, 62, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 60, 60, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 60, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,411 (224.26 KB)

 Trainable params: 56,771 (221.76 KB)

 Non-trainable params: 640 (2.50 KB)

Model Created Successfully!


### **Check Model’s Structure:**
Using the **plot_model** function you can check the structure of the final model, this is really helpful when you’re creating a complex network and you want to make sure you have constructed the network correctly.

In [39]:
plot_model(model, to_file = 'model_structure_plot.png', show_shapes = True, show_layer_names = True)

You must install pydot (`pip install pydot`) for `plot_model` to work.


## **Step 6: Compile & Train the Model**


Now let's start the training. Before we do that, we also need to complile the model.


In [40]:
# Adding the Early Stopping Callback to the model which will continuously monitor the validation loss metric for every epoch.
# If the models validation loss does not decrease after 15 consecutive epochs, the training will be stopped and the weight which reported the lowest validation loss will be retored in the model.
early_stopping_callback = EarlyStopping(monitor = 'val_loss', patience = 15, mode = 'min', restore_best_weights = True)

# Adding loss, optimizer and metrics values to the model.
model.compile(loss = 'categorical_crossentropy', optimizer = 'Adam', metrics = ["accuracy"])

# Start Training
model_training_history = model.fit(x = features_train, y = labels_train, epochs = 20, batch_size = 4 , shuffle = True, validation_split = 0.2, callbacks = [early_stopping_callback])

Epoch 1/20
480/480 ━━━━━━━━━━━━━━━━━━━━ 43s 82ms/step - accuracy: 0.5453 - loss: 0.9638 - val_accuracy: 0.3771 - val_loss: 1.7741
Epoch 2/20
480/480 ━━━━━━━━━━━━━━━━━━━━ 41s 85ms/step - accuracy: 0.5964 - loss: 0.8718 - val_accuracy: 0.3812 - val_loss: 3.6873
Epoch 3/20
480/480 ━━━━━━━━━━━━━━━━━━━━ 36s 75ms/step - accuracy: 0.6500 - loss: 0.8091 - val_accuracy: 0.3917 - val_loss: 3.6971
Epoch 4/20
480/480 ━━━━━━━━━━━━━━━━━━━━ 42s 77ms/step - accuracy: 0.6802 - loss: 0.7384 - val_accuracy: 0.7917 - val_loss: 0.6126
Epoch 5/20
480/480 ━━━━━━━━━━━━━━━━━━━━ 37s 78ms/step - accuracy: 0.7182 - loss: 0.6826 - val_accuracy: 0.4583 - val_loss: 2.2640
Epoch 6/20
480/480 ━━━━━━━━━━━━━━━━━━━━ 38s 78ms/step - accuracy: 0.7323 - loss: 0.6337 - val_accuracy: 0.8521 - val_loss: 0.3954
Epoch 7/20
480/480 ━━━━━━━━━━━━━━━━━━━━ 38s 79ms/step - accuracy: 0.7651 - loss: 0.5719 - val_accuracy: 0.7833 - val_loss: 0.4553
Epoch 8/20
480/480 ━━━━━━━━━━━━━━━━━━━━ 37s 78ms/step - accuracy: 0.7880 - loss: 0.5130 - 

### **Evaluating Your Trained Model**
Evaluate your trained model on the feature's and label's test sets.

In [41]:
model_evaluation_history = model.evaluate(features_test, labels_test)

19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 82ms/step - accuracy: 0.9300 - loss: 0.2225


### **Save Your Model**
You should now save your model for future runs.

In [42]:
"""
# Creating a useful name for our model, incase you're saving multiple models (OPTIONAL)
date_time_format = '%Y_%m_%d__%H_%M_%S'
current_date_time_dt = dt.datetime.now()
current_date_time_string = dt.datetime.strftime(current_date_time_dt, date_time_format)
model_evaluation_loss, model_evaluation_accuracy = model_evaluation_history
model_name = f'Model___Date_Time_{current_date_time_string}___Loss_{model_evaluation_loss}___Accuracy_{model_evaluation_accuracy}.h5'

# Saving your Model
model.save(model_name)
"""

student_name = "Anfal Bamardouf" # Replace with your actual name
save_path = f"{student_name}_ucf11_model.h5"
model.save(save_path)
print(f"Model saved as {save_path}")

Model saved as Anfal Bamardouf_ucf11_model.h5


## **Step 9: Using Single-Frame CNN Method:**
Now let's create a function that will output a singular prediction for the complete video, now this function will take `n` frames from the entire video and make predictions. In the end, it will average the predictions of those <code>n</code> frames to give you the final activity class for that video. You can set the value of <code>n</code> using the <code>predictions_frames_count</code> variable.

This function is useful when you have a video containing one activity and you want to know the activity's name and its score.


In [23]:
def make_average_predictions(video_file_path, predictions_frames_count):
    
    # Initializing the Numpy array which will store Prediction Probabilities
    predicted_labels_probabilities_np = np.zeros((predictions_frames_count, model_output_size), dtype = float)

    # Reading the Video File using the VideoCapture Object
    video_reader = cv2.VideoCapture(video_file_path)

    # Getting The Total Frames present in the video 
    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))

    # Calculating The Number of Frames to skip Before reading a frame
    skip_frames_window = video_frames_count // predictions_frames_count

    for frame_counter in range(predictions_frames_count): 

        # Setting Frame Position
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)

        # Reading The Frame
        _ , frame = video_reader.read() 

        # Resize the Frame to fixed Dimensions
        resized_frame = cv2.resize(frame, (image_height, image_width))
        
        # Normalize the resized frame by dividing it with 255 so that each pixel value then lies between 0 and 1
        normalized_frame = resized_frame / 255

        # Passing the Image Normalized Frame to the model and receiving Predicted Probabilities.
        predicted_labels_probabilities = model.predict(np.expand_dims(normalized_frame, axis = 0),verbose=0)[0]

        # Appending predicted label probabilities to the deque object
        predicted_labels_probabilities_np[frame_counter] = predicted_labels_probabilities

    # Calculating Average of Predicted Labels Probabilities Column Wise 
    predicted_labels_probabilities_averaged = predicted_labels_probabilities_np.mean(axis = 0)

    # Sorting the Averaged Predicted Labels Probabilities
    predicted_labels_probabilities_averaged_sorted_indexes = np.argsort(predicted_labels_probabilities_averaged)[::-1]

    # Iterating Over All Averaged Predicted Label Probabilities
    for predicted_label in predicted_labels_probabilities_averaged_sorted_indexes:

        # Accessing The Class Name using predicted label.
        predicted_class_name = classes_list[predicted_label]

        # Accessing The Averaged Probability using predicted label.
        predicted_probability = predicted_labels_probabilities_averaged[predicted_label]

        print(f"CLASS NAME: {predicted_class_name:<15} AVERAGED PROBABILITY: {predicted_probability:.2f}")    

    # Closing the VideoCapture Object and releasing all resources held by it. 
    video_reader.release()

In [43]:
test_videos = {
    "walking": "/Users/anfalbamardouf/Downloads/HW_CV_lab/arti560-computer-vision-labs/lab07-action-recognition/Youtube Videos/walking.mp4",
    "basketball": "/Users/anfalbamardouf/Downloads/HW_CV_lab/arti560-computer-vision-labs/lab07-action-recognition/Youtube Videos/basketball.mp4",
    "trampoline_jumping": "/Users/anfalbamardouf/Downloads/HW_CV_lab/arti560-computer-vision-labs/lab07-action-recognition/Youtube Videos/trampoline_jumping.mp4"
}

for true_class, video_path in test_videos.items():
    print("\nActual class:", true_class)
    print("Video path:", video_path)
    make_average_predictions(video_path, 50)


Actual class: walking
Video path: /Users/anfalbamardouf/Downloads/HW_CV_lab/arti560-computer-vision-labs/lab07-action-recognition/Youtube Videos/walking.mp4
CLASS NAME: trampoline_jumping AVERAGED PROBABILITY: 0.72
CLASS NAME: walking         AVERAGED PROBABILITY: 0.28
CLASS NAME: basketball      AVERAGED PROBABILITY: 0.01

Actual class: basketball
Video path: /Users/anfalbamardouf/Downloads/HW_CV_lab/arti560-computer-vision-labs/lab07-action-recognition/Youtube Videos/basketball.mp4
CLASS NAME: basketball      AVERAGED PROBABILITY: 0.49
CLASS NAME: trampoline_jumping AVERAGED PROBABILITY: 0.32
CLASS NAME: walking         AVERAGED PROBABILITY: 0.19

Actual class: trampoline_jumping
Video path: /Users/anfalbamardouf/Downloads/HW_CV_lab/arti560-computer-vision-labs/lab07-action-recognition/Youtube Videos/trampoline_jumping.mp4
CLASS NAME: trampoline_jumping AVERAGED PROBABILITY: 0.48
CLASS NAME: walking         AVERAGED PROBABILITY: 0.37
CLASS NAME: basketball      AVERAGED PROBABILITY: